# 14 · Condition-specialist autoencoder bank for unfamiliar faults (Stage R4)

## Purpose
Can reconstruction by **per-condition specialists** recognise recordings of a fault class the system never trained on
better than the current **Mahalanobis distance** on the CNN-6 embedding? In each of the 27 leave-one-class-out runs of
Stage 3C (9 withheld classes × 3 outer folds, seed 42), nine autoencoders are used: normal (the run's saved Stage 3C
AE-6) and one per remaining fault class, each trained only on its own class's fit recordings with the AE-6 recipe and its
own normalisation. The withheld class has no specialist and never reaches training, normalisation or calibration.
Protocol declared in CLAUDE.md §7 (Stage R4) before training; exploratory development data, **seed 42 only** (a stated
limitation: one training seed); reserved tests sealed.

**Scores and rule.** Each specialist's recording error e_u is made comparable by s_u = log(e_u / median of its own class's
calibration errors), leave-one-out for those recordings. (`ml.openset.empirical_rank` was not used as primary: it is capped
at n/(n+1), and normal has only 2–3 calibration recordings per run.) S = min_u s_u; τ_S = the 97.5 % order statistic of S
over all calibration recordings; specialist u accepts iff s_u ≤ τ_S. No acceptance → potentially unfamiliar; several →
ambiguous, and the whole accepting set is kept.

## Inputs
`metadata.csv`, `results/dev_folds.csv`, `results/q1q3/q3_fit_cal_eval_assignment.csv`; `results/research/r4_specialists.json`,
`r4_per_run.csv`, `r4_ambiguity.csv` (per instance), `r4_generalisation.csv`, `r4_time_estimate.json` (written by
`scripts/34_train_specialists.py`); `results/specialists/w-*/fold*_train.json`. The comparison scores are the saved Stage 3C
Mahalanobis distances (`results/scores/cnn6ae6_s42_loco_decisions.parquet`, τ_d from `results/loco/decision_thresholds.csv`).
Config and split hashes are printed below.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the configuration, the development pool and the saved R4 outputs, prints the config and split hashes, and checks that all 27 runs are present with nine specialists each and no specialist for the withheld class.</div>

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.config import load_config, config_sha256
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR
from ml.splits import FAULT_CLASSES, development_pool, split_hash

ROOT = SYSTEM_CODE_DIR
R = ROOT / "results"
cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(ROOT / "metadata.csv")
pool = development_pool(meta, cfg.development.allowed_closed, cfg.development.reserved_columns)
assert len(pool) == 804, "development pool size differs: stop and ask"
print("config sha256:", config_sha256(cfg)[:16], "| split hash:", split_hash(meta)[:16])
res = json.loads((R / "research/r4_specialists.json").read_text())
runs = pd.read_csv(R / "research/r4_per_run.csv")
inst = pd.read_csv(R / "research/r4_ambiguity.csv", keep_default_na=False, na_values=[""])
gen = pd.read_csv(R / "research/r4_generalisation.csv")
trains = [json.loads(p.read_text()) for p in sorted((R / "specialists").glob("w-*/fold*_train.json"))]
assert len(runs) == 27 and len(trains) == 27
assert all(len(t["specialists"]) == 9 and t["withheld"] not in t["specialists"] for t in trains)
print("runs:", len(runs), "| instances:", len(inst), "| roles:", inst.role.value_counts().to_dict())

## Training set sizes and cost
Per specialist: fit (training) and calibration recordings across the 27 runs. The normal specialist is the Stage 3C AE-6
(reused; the reuse reproduced its saved errors). Nine models is a real deployment cost, so size and latency are reported
next to the single Mahalanobis check.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> tabulates, per specialist class, the smallest and largest number of fit and calibration recordings over the runs, then prints the training time, the projected time after the first run, the parameter counts and the CPU latency per recording of the bank and of CNN-6 + Mahalanobis.</div>

In [ ]:
sizes = pd.DataFrame([{"specialist": u, **v} for t in trains for u, v in t["specialists"].items()])
print(sizes.groupby("specialist")[["n_fit_recordings", "n_cal_recordings"]].agg(["min", "max"]).to_string())
c = res["cost"]
print(f"specialist training: {c['specialist_training_seconds_total'] / 60:.1f} min for {c['specialists_trained']} specialists "
      f"({c['normal_reused_from_3c']} normal specialists reused from 3C); projected after the first run: "
      f"{res['time_estimate']['projected_minutes']:.0f} min")
print("parameters:", c["parameters"]); print("CPU ms per recording:", {t: {k: round(v, 1) for k, v in d.items()} for t, d in c["cpu_ms_per_recording"].items()})
print("reused AE-6 reproduction, max |diff|:", res["normal_reproduction_max_abs_diff"])

## Primary: threshold-free separation per fold (AUROC, OSCR)
Known side = the unseen outer fold's known recordings (normals included); unknown side = the withheld class in that fold.
Per fold, the mean over its 9 runs; then mean and spread across folds (§5 rule 13). OSCR uses the Stage 3C CNN-6 top class
as the closed-set prediction for both scores, so the two differ only in the unfamiliarity score.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the fold means of AUROC and OSCR for the bank and for Mahalanobis, the per-fold AUROC difference (bank − Mahalanobis), and the same numbers per withheld class (mean over its three folds) with the number of folds in which the bank is higher.</div>

In [ ]:
fold = pd.DataFrame(res["primary_per_fold"]).T
fold.index.name = "outer fold"
print(fold.round(4).to_string())
print("Δ AUROC per fold:", {k: round(v, 4) for k, v in res["delta_auroc_per_fold"].items()}, "| mean", round(res["mean_delta_auroc"], 4))
pc = pd.DataFrame(res["per_withheld_class"]).T
pc["folds_bank_higher"] = pd.Series(res["runs_bank_better_per_class"])
pc["Δ AUROC"] = pc.auroc_bank - pc.auroc_maha
print(pc.sort_values("Δ AUROC").round(4).to_string())

## Unknown recall at a matched known-side rejection count
Per run, the bank's threshold is placed to reject exactly as many known recordings as Mahalanobis rejects at its calibrated
τ_d; then the share of withheld recordings each rejects. Pooled over runs (instances) and per fold; also both at their own
calibrated thresholds.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints pooled and per-fold unknown recall at the matched known-side rejection count, the calibrated rates of both checks, and draws per withheld class the matched unknown recall of the bank against Mahalanobis (one dot per fold, the bar is the mean). The figure is saved for the report.</div>

In [ ]:
m = res["matched_unknown_recall"]
print(f"pooled: bank {m['bank']:.3f} vs Mahalanobis {m['mahalanobis']:.3f}")
print("per fold:", {k: {a: round(b, 3) for a, b in v.items()} for k, v in m["per_fold"].items()})
print("calibrated:", {k: round(v, 3) for k, v in res["calibrated"].items()})
order = pc.sort_values("Δ AUROC").index.tolist()
fig, ax = plt.subplots(figsize=(11, 4.6))
xx = np.arange(len(order))
for off, col, lab, key in ((-0.2, "#4f6bff", "specialist bank (matched)", "bank_matched_unknown_recall"),
                           (0.2, "#9e9e9e", "Mahalanobis at τ_d", "maha_unknown_recall")):
    means = [runs[runs.withheld == c][key].mean() for c in order]
    ax.bar(xx + off, means, width=0.38, color=col, alpha=0.75, label=lab)
    for i, c in enumerate(order):
        v = runs[runs.withheld == c][key]
        ax.scatter(np.full(len(v), i + off), v, color="k", s=10, zorder=3)
ax.set_xticks(xx, [c.replace("_", " ") for c in order], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("withheld recordings rejected (share)"); ax.set_ylim(0, 1.05)
ax.set_title("Unknown recall per withheld class at the same known-side rejection count (dots = outer folds; seed 42)", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(R / "figures/r4_unknown_recall.png", dpi=150); plt.show()

## Unfamiliar faults called healthy vs called a known fault
Two different failures: a withheld fault accepted as **healthy** (the machine is let go) and accepted as a **known fault**
(the wrong repair is suggested). Bank verdicts at τ_S; Mahalanobis-only (accepted → the CNN-6 top class); rule B of Stage 3C
(MSP and Mahalanobis) for context.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> counts the verdicts on the withheld recordings (instances over the 27 runs) for the bank, for the Mahalanobis check on its own and for the full Stage 3C rule B, and per withheld class the share called healthy by each.</div>

In [ ]:
print(json.dumps(res["unknown_outcomes"], indent=1))
un = inst[inst.role == "foldk_withheld"]
t = pd.DataFrame({"bank healthy": un.groupby("withheld").bank_verdict.apply(lambda v: (v == "healthy").mean()),
                  "bank ambiguous healthy/fault": un.groupby("withheld").bank_verdict.apply(lambda v: (v == "ambiguous_healthy_fault").mean()),
                  "Mahalanobis-only healthy": un.groupby("withheld").maha_only_verdict.apply(lambda v: (v == "healthy").mean()),
                  "n": un.groupby("withheld").size()})
print(t.round(3).to_string())

## Do the specialists reconstruct the withheld class? (generalisation check)
The most likely failure: autoencoders reconstruct inputs they never saw. Per specialist, its acceptance of withheld
recordings against its acceptance of the other known classes that are not its own ("foreign"); which specialist is the
most compatible for each withheld class; how often the normal specialist accepts a withheld fault; and whether low scores
go with low input level.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints each specialist's mean acceptance of its own class, of foreign known classes and of the withheld class, the share of withheld recordings accepted by the normal specialist, the rank correlation of the bank score with the recording's input level, and draws a heatmap of the median specialist score on each withheld class (low = reconstructed as well as that specialist's own calibration recordings). The figure is saved for the report.</div>

In [ ]:
g = res["generalisation"]
print(pd.DataFrame(g["per_specialist_mean"]).T.round(3).to_string())
print(f"withheld accepted by the normal specialist: {g['withheld_accepted_by_normal_specialist']:.3f}")
print(f"Spearman(S, input level): withheld {g['spearman_S_vs_input_level_withheld']:+.3f}, known {g['spearman_S_vs_input_level_known']:+.3f}")
best = un.groupby("withheld").best.value_counts(normalize=True).rename("share").reset_index()
print(best.sort_values(["withheld", "share"], ascending=[True, False]).groupby("withheld").head(2).round(3).to_string(index=False))
H = gen.groupby(["withheld", "specialist"]).withheld_median_s.mean().unstack().reindex(index=FAULT_CLASSES, columns=CLASS10)
fig, ax = plt.subplots(figsize=(9.5, 6))
im = ax.imshow(H.to_numpy(dtype=float), cmap="RdBu", vmin=-3, vmax=3)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.iat[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7)
ax.set_xticks(range(10), [c.replace("_", " ") for c in CLASS10], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(9), [c.replace("_", " ") for c in FAULT_CLASSES], fontsize=8)
ax.set_xlabel("specialist"); ax.set_ylabel("withheld class (its recordings)")
fig.colorbar(im, ax=ax, label="median s_u = log(error / own calibration median)")
ax.set_title("How well each specialist reconstructs the withheld class (blank = no specialist; seed 42)", fontsize=10)
fig.tight_layout(); fig.savefig(R / "figures/r4_specialist_scores.png", dpi=150); plt.show()

## Ambiguity: several specialists accept
How often more than one specialist accepts a recording, by role and class, which combinations occur, and how it relates to
speed.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the ambiguity share by role, by class on the known side, the most frequent accepting sets among ambiguous recordings, and the ambiguity share by speed band; then the ambiguity share by severity for the known side.</div>

In [ ]:
a = res["ambiguity"]
print(pd.DataFrame(a["by_role"]).T.round(3).to_string())
print("known side by class:", a["known_side_by_class"])
print("most frequent accepting sets:", a["top_sets"])
print("by speed band:", a["by_speed_band"])
kn = inst[inst.role == "foldk_known"]
print(kn.assign(amb=kn.n_accepting >= 2).groupby(["class10", "severity"]).amb.mean().round(3).unstack().to_string())

## Speed and severity within the development pool
Unknown recall and known-side rejection at the calibrated thresholds by speed band; unknown recall by the withheld
recording's severity.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the calibrated unknown recall and known-side rejection of both checks per speed band, then the unknown recall per withheld class and severity level with the number of instances.</div>

In [ ]:
print(pd.DataFrame(res["speed"]).round(3).to_string())
sv = pd.DataFrame(res["severity_unknown_recall"])
print(sv.round(3).to_string(index=False))

## Pre-declared reading
**Helps** only if (a) Δ AUROC > 0 in all 3 folds, (b) mean Δ ≥ +0.02, (c) pooled unknown recall at the matched known-side
rejection count ≥ Mahalanobis + 5 points, (d) per-class mean AUROC higher for ≥ 5 of 9 withheld classes. **Does not help**
if Δ ≤ 0 in all folds or mean Δ ≤ −0.02. Otherwise **too thin to tell**. Per class: "helps for X" only if higher in all 3
folds of X.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the four criteria, the reading, and the withheld classes for which the bank's AUROC is higher in all three folds.</div>

In [ ]:
print(res["criteria"]); print("READING:", res["reading"])
print("helps for (all 3 folds higher):", [c for c, n in res["runs_bank_better_per_class"].items() if n == 3] or "none")

## Findings

Exploratory development results, **seed 42 only** (one training seed; the 27 Stage 3C LOCO runs), declared before training.

- **Pre-declared reading: "too thin to tell"; the bank is a partial, class-specific improvement, not a general one.**
  Threshold-free AUROC is higher in all 3 folds (bank 0.921 / 0.945 / 0.873 vs Mahalanobis 0.870 / 0.893 / 0.868; Δ +0.051 /
  +0.052 / +0.005, mean +0.036; OSCR 0.912 / 0.934 / 0.866 vs 0.862 / 0.886 / 0.863), and it is higher for 5 of 9 withheld
  classes, but criterion (c) fails: at a matched known-side rejection count, pooled unknown recall is **0.583 vs 0.582**
  (per fold 0.760 vs 0.624, 0.458 vs 0.579, 0.536 vs 0.540).
- **Where it helps and where it hurts.** Higher in all 3 folds for withheld **vertical misalignment (AUROC 0.964 vs 0.454)**
  and **imbalance (0.881 vs 0.637)**, the pair the CNN-6 embedding cannot separate, and for the three ball / underhang-outer
  classes (already ≥ 0.95). Much worse for **horizontal misalignment (0.718 vs 0.949)**, overhang outer (0.856 vs 0.985),
  overhang cage (0.895 vs 0.969) and underhang cage (0.901 vs 0.968). The mean gain comes largely from the two classes
  where Mahalanobis is near chance.
- **The likely failure mode is real: the normal specialist accepts unfamiliar faults.** At τ_S the bank calls **92 withheld
  instances healthy and 78 "healthy or fault"** (of 779; 22 % accepted by the normal specialist), against **19 healthy** for
  Mahalanobis alone and 28 for rule B. Withheld horizontal misalignment is called healthy in 64 % of instances (bank) vs 21 %
  (Mahalanobis); overhang cage 31 % vs 0 %. The normal specialist is the most compatible one for 90 % of withheld horizontal
  misalignment and 49–60 % of withheld cage / overhang-outer recordings. Its acceptance of the withheld class (0.25) is the
  same as of foreign known classes (0.22), and S rises with input level (Spearman +0.60 on withheld, +0.43 on known): low-
  level recordings are reconstructed well by a specialist that never saw them. This is generalisation of the autoencoders,
  not evidence of resemblance to a known condition. Where the bank succeeds (imbalance ↔ vertical misalignment), the most
  compatible specialist is the partner class (vertical misalignment for 98 % of withheld imbalance, imbalance for 85 % of
  withheld vertical misalignment), yet its error is still above τ_S, which is what separates them.
- **Ambiguity is common on known recordings and rare on unknowns:** 31–34 % of known instances are accepted by ≥ 2
  specialists (mean 1.3–1.4 accepting) vs 11 % of withheld instances. Most frequent sets: horizontal misalignment + normal
  (762), imbalance + vertical misalignment (684), normal + overhang outer (396), normal + underhang cage (280). Normal
  recordings are ambiguous in 78 % of instances; mild settings dominate (horizontal misalignment 0.5 mm 0.92, imbalance
  6–10 g 0.66–0.70 vs 25–30 g ≤ 0.06); ambiguity is lower at ≥ 30 Hz (0.25) than below (0.33–0.34).
- **Speed and severity:** at the calibrated thresholds the bank's unknown recall is higher in every speed band (0.66 /
  0.68 / 0.71 vs 0.60 / 0.58 / 0.57) but its known-side rejection is higher at ≥ 30 Hz (0.157 vs 0.093); overall 0.107 vs
  0.094 known rejection for 0.683 vs 0.582 unknown recall (unmatched rates, not the criterion). By severity the bank rejects
  severe imbalance (20–30 g: 0.96–1.00) but not mild (6 g: 0.00), and mild bearing-cage / outer settings are its weak spot.
- **Cost:** 216 specialists trained in 76.4 min (projected 72 min after the first run; the 27 normal specialists reused);
  881,478 parameters for the nine (97,942 each) vs 601,545 for the LOCO CNN-6, which the system runs anyway; CPU latency per
  recording **107–118 ms for the bank vs 4–14 ms for CNN-6 + Mahalanobis** (default threads / one thread), an additional cost
  in any deployment. **What this does and does not show:** a reconstruction bank separates the one confusable pair the
  embedding cannot, but on these folds it trades that for letting mild and low-level unfamiliar faults through as healthy;
  one seed, and nothing here is selected or deployed.


## Outputs written
- By `scripts/34_train_specialists.py`: `results/research/r4_specialists.json`, `r4_per_run.csv`, `r4_ambiguity.csv`,
  `r4_generalisation.csv`, `r4_time_estimate.json`; `results/specialists/w-*/fold*_train.json` and `fold*_<class>_norm.json`;
  `results/scores/specialists_w-*_s42_loco_fold*_errors.parquet`; weights in `artifacts/specialists/` (gitignored).
- By this notebook: `results/figures/r4_unknown_recall.png`, `results/figures/r4_specialist_scores.png`.